## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 🛑 Middleware Termination

## Industry Use Case: Transaction Compliance Screening

This notebook demonstrates how middleware can **terminate** the execution pipeline early.

| Feature | FSI Application |
|---------|-----------------|
| **Pre-Termination** | Block prohibited transaction types |
| **Post-Termination** | Rate limit transaction requests |
| **Short-Circuit** | Return immediate compliance responses |

In [1]:
import os
from dotenv import load_dotenv

load_dotenv('../../.env', override=True)

print(f"✅ Environment loaded")

✅ Environment loaded


In [2]:
from collections.abc import Awaitable, Callable

from agent_framework import (
    AgentMiddleware,
    AgentContext,
    AgentResponse,
    Message,
    Role,
)
from agent_framework_foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

print("✅ All imports loaded")

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


✅ All imports loaded


## Define Transaction Tool Function

In [3]:
def check_transaction(transaction_id: str) -> str:
    """Check the status of a transaction."""
    return f"Transaction {transaction_id} is approved and completed."

print("✅ Tool defined: check_transaction")

✅ Tool defined: check_transaction


## Termination Middleware Classes

**PreTerminationMiddleware**: Blocks requests with prohibited transaction types BEFORE processing

**PostTerminationMiddleware**: Allows processing but limits total requests (rate limiting)

In [4]:
class PreTerminationMiddleware(AgentMiddleware):
    """Terminates execution BEFORE agent processing for prohibited terms."""

    def __init__(self, blocked_words: list[str]):
        self.blocked_words = [word.lower() for word in blocked_words]

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        last_message = context.messages[-1] if context.messages else None
        if last_message and last_message.text:
            query = last_message.text.lower()
            for blocked_word in self.blocked_words:
                if blocked_word in query:
                    print(f"[PreTermination] ⛔ Blocked: '{blocked_word}' detected")
                    context.result = AgentResponse(
                        messages=[
                            Message(
                                role="assistant",
                                contents=[f"Cannot process requests containing '{blocked_word}'. Please contact compliance."],
                            )
                        ]
                    )
                    context.terminate = True
                    return  # Don't call next()

        await call_next()

class PostTerminationMiddleware(AgentMiddleware):
    """Allows processing but terminates after max responses (rate limiting)."""

    def __init__(self, max_responses: int = 2):
        self.max_responses = max_responses
        self.response_count = 0

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        if self.response_count >= self.max_responses:
            print(f"[PostTermination] ⛔ Rate limit reached ({self.max_responses} requests)")
            context.terminate = True
            return

        await call_next()
        self.response_count += 1
        print(f"[PostTermination] Request {self.response_count}/{self.max_responses}")

print("✅ Middleware classes defined")

✅ Middleware classes defined


## Example 1: Pre-termination (Compliance Blocking)

Blocks requests containing prohibited transaction types.

In [5]:
async def run_pre_termination_example():
    """Pre-termination blocks prohibited requests."""
    print("\n--- Example 1: Pre-termination Middleware ---\n")
    
    async with (
        AzureCliCredential() as credential,
        FoundryChatClient(credential=credential).as_agent(
            name="TransactionAgent",
            instructions="You help with transaction inquiries.",
            tools=[check_transaction],
            middleware=[PreTerminationMiddleware(blocked_words=["offshore", "anonymous"])],
        ) as agent,
    ):
        # Normal query
        print("1. Normal query:")
        result = await agent.run("Check transaction TXN-12345")
        print(f"   Agent: {result.text}\n")

        # Blocked query
        print("2. Blocked query (contains 'offshore'):")
        result = await agent.run("How do I set up offshore transfers?")
        print(f"   Agent: {result.text}\n")

await run_pre_termination_example()


--- Example 1: Pre-termination Middleware ---



1. Normal query:


   Agent: Transaction TXN-12345 is approved and completed.

2. Blocked query (contains 'offshore'):
[PreTermination] ⛔ Blocked: 'offshore' detected
   Agent: Cannot process requests containing 'offshore'. Please contact compliance.



## Example 2: Post-termination (Rate Limiting)

Allows initial requests but terminates after max responses.

In [6]:
async def run_post_termination_example():
    """Post-termination limits requests after max responses."""
    print("\n--- Example 2: Post-termination Middleware ---\n")
    
    async with (
        AzureCliCredential() as credential,
        FoundryChatClient(credential=credential).as_agent(
            name="TransactionAgent",
            instructions="You help with transaction inquiries.",
            tools=[check_transaction],
            middleware=[PostTerminationMiddleware(max_responses=2)],
        ) as agent,
    ):
        # First request (allowed)
        print("1. First request:")
        result = await agent.run("Check transaction TXN-001")
        print(f"   Agent: {result.text if result and result.text else 'No response'}\n")

        # Second request (allowed)
        print("2. Second request:")
        result = await agent.run("Check transaction TXN-002")
        print(f"   Agent: {result.text if result and result.text else 'No response'}\n")

        # Third request (blocked by rate limit)
        print("3. Third request (rate limited):")
        result = await agent.run("Check transaction TXN-003")
        print(f"   Agent: {result.text if result and result.text else 'No response (rate limited)'}\n")

await run_post_termination_example()


--- Example 2: Post-termination Middleware ---



1. First request:


[PostTermination] Request 1/2
   Agent: Transaction TXN-001 has been approved and completed.

2. Second request:


[PostTermination] Request 2/2
   Agent: Transaction **TXN-002** is approved and completed.

3. Third request (rate limited):
[PostTermination] ⛔ Rate limit reached (2 requests)
   Agent: No response (rate limited)



## Key Takeaways

| Technique | When to Use | FSI Example |
|-----------|-------------|-------------|
| `context.terminate = True` + `return` | Block before processing | Prohibited transaction types |
| `await next()` then `terminate` | Allow then limit | Rate limiting |
| Set `context.result` | Custom response | Compliance messages |

## Next Steps
- **Override Result** (notebook 8) - Modify agent responses
- **Shared State** (notebook 9) - Cross-middleware communication